# 第7章 概率、分布与抽样

> **核心问题**：不确定结果如何用模型表达？样本平均为什么会变化？正态分布为什么有用又危险？

- 金融线：情景、损失概率、尾部事件和模型风险。
- 数学线：随机变量、期望、方差、分位数、大数定律和中心极限定理。
- Python线：随机生成器、布尔统计、直方图、经验分布、重复抽样和动画。

## AI学习状态

当前进度：第7章开始  
已掌握：收益率和财富路径  
仍然薄弱：待填写  
下一步：区分总体假设、一次样本和估计结果。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"]=(8,4.5); plt.rcParams["axes.grid"]=True
plt.rcParams["font.sans-serif"]=["Arial Unicode MS","PingFang SC","SimHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"]=False
rng=np.random.default_rng(20260711)

## 7.1 随机变量：给结果赋数值

设一年后收益 $R$有三个情景：-20%、5%、30%，概率分别为0.2、0.5、0.3。概率必须非负且总和为1。

$$E[R]=\sum_i p_ir_i,\qquad Var(R)=\sum_i p_i(r_i-E[R])^2$$

In [ ]:
outcomes=np.array([-.20,.05,.30]); probabilities=np.array([.2,.5,.3])
expected=np.sum(outcomes*probabilities)
variance=np.sum(probabilities*(outcomes-expected)**2)
print({"概率和":probabilities.sum(),"期望收益":f"{expected:.2%}","标准差":f"{np.sqrt(variance):.2%}"})

### 我的解释

期望收益是否一定是三个情景之一？标准差为什么与收益率使用相同单位，而方差不是？

<!-- 在这里填写；完成前AI不要代答 -->

## 7.2 模拟不是“制造事实”

随机模拟从我们指定的概率模型中抽样。它能回答“如果模型成立会怎样”，不能证明模型符合市场。

In [ ]:
draws=rng.choice(outcomes,size=10_000,p=probabilities)
print({"模拟均值":draws.mean(),"理论期望":expected,"模拟亏损比例":np.mean(draws<0)})
plt.hist(draws,bins=[-.25,-.075,.175,.35],rwidth=.8); plt.xlabel("收益情景"); plt.ylabel("次数"); plt.title("离散情景的10000次抽样"); plt.show()

## 7.3 大数定律：样本平均逐渐稳定

大数定律不保证短期接近期望，也不说明期望本身安全；它说明在适当条件下，独立同分布样本平均随样本量增加趋近总体期望。

In [ ]:
draws=rng.choice(outcomes,size=5000,p=probabilities)
running_mean=np.cumsum(draws)/np.arange(1,len(draws)+1)
plt.plot(running_mean,label="累计样本均值"); plt.axhline(expected,color="red",ls="--",label="理论期望")
plt.xscale("log"); plt.xlabel("样本量（对数轴）"); plt.ylabel("平均收益"); plt.title("大数定律的数值观察"); plt.legend(); plt.show()

### 观察问题

为什么前几十次波动剧烈？把横轴改为线性后，视觉感受有何变化？金融市场收益为何可能不满足独立同分布？

### 我的回答

<!-- 在这里填写；完成前AI不要代答 -->

## 7.4 分位数与尾部

5%分位数表示约5%的观察不高于该数，并不等于“最大损失”。经验分位数依赖样本和方法，在小样本下尤其不稳定。

In [ ]:
normal=rng.normal(.0003,.01,100_000)
heavy=rng.standard_t(df=4,size=100_000)*.01/np.sqrt(4/(4-2))+.0003
rows=[]
for name,x in [("正态",normal),("t(4)厚尾",heavy)]:
    rows.append({"模型":name,"均值":x.mean(),"标准差":x.std(ddof=1),"1%分位":np.quantile(x,.01),"绝对收益>4%":np.mean(np.abs(x)>.04)})
display(pd.DataFrame(rows).set_index("模型").style.format("{:.3%}"))

In [ ]:
fig,ax=plt.subplots(); bins=np.linspace(-.06,.06,120)
ax.hist(normal,bins=bins,density=True,alpha=.5,label="正态"); ax.hist(heavy,bins=bins,density=True,alpha=.5,label="t(4)厚尾")
ax.set_yscale("log"); ax.set(title="相同均值和方差附近、不同尾部（纵轴对数）",xlabel="收益",ylabel="密度"); ax.legend(); plt.show()

**量化编程警告**：样本均值和标准差相近，不代表极端风险相近。正态模型不是默认真理；模型选择应结合机制、诊断和压力测试。

## 7.5 抽样分布与中心极限定理

重复抽取大小为$n$的样本，每次计算均值；这些均值本身构成抽样分布。许多条件下，样本均值标准误约为`总体标准差/sqrt(n)`。

In [ ]:
population=rng.standard_t(df=4,size=500_000)
fig,axes=plt.subplots(1,3,figsize=(13,3.8))
for ax,n in zip(axes,[1,10,100]):
    means=rng.choice(population,size=(5000,n),replace=True).mean(axis=1)
    ax.hist(means,bins=50,density=True); ax.set_title(f"样本量 n={n}"); ax.set_xlabel("样本均值")
axes[0].set_ylabel("密度"); fig.suptitle("样本均值的抽样分布"); plt.tight_layout(); plt.show()

### 我的解释

随着$n$增加，样本均值分布的中心和宽度如何变化？这是否说明单日收益本身变成正态？

<!-- 在这里填写；完成前AI不要代答 -->

## 7.6 编程练习：离散分布统计

检查概率合法性，返回期望、方差和标准差。

In [ ]:
def discrete_stats(outcomes,probabilities):
    # TODO
    return None

In [ ]:
ans=discrete_stats([-1,1],[.5,.5])
if ans is None: print("练习尚未完成。")
else: print("基础测试：",np.allclose(ans,[0,1,1]))

## 本章总结与小项目

自行设计一个三情景投资模型，计算理论统计量；模拟不同样本量；比较正态与厚尾模型的1%分位和极端事件比例；解释模拟结论依赖哪些假设。

**关键区分**：总体、样本、估计量和模拟输出不是同一个对象。